# 02 · Silver layer check

Run after `build_silver.py` (README, Phase 4). Same kernel setup as notebook 00.

You'll see: Bronze vs Silver row counts (where the duplicates went), the typed schema, the
data quality results, and the analysis-ready `order_items_enriched` table answering real questions.

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/src')

from pyspark.sql import functions as F
from common.spark_session import get_spark, lake_path

spark = get_spark('silver-check')
bronze = lambda t: spark.read.format('delta').load(lake_path('bronze', t))
silver = lambda t: spark.read.format('delta').load(lake_path('silver', t))

## 1. Bronze vs Silver row counts

`orders` drops by 600: the duplicate test sends. `geolocation` collapses 1M raw points to one per zip prefix.

In [ ]:
for t in ['orders', 'customers', 'order_items', 'order_payments', 'order_reviews',
          'products', 'sellers', 'geolocation']:
    print(f"{t:<16} bronze {bronze(t).count():>10,}   silver {silver(t).count():>10,}")
print(f"{'order_items_enriched':<16}        {'':>10}   silver {silver('order_items_enriched').count():>10,}")

## 2. Real types now, not strings

Compare with notebook 01: timestamps, ints and `decimal(12,2)` money.

In [ ]:
silver('orders').printSchema()
silver('order_items').select('order_id', 'price', 'freight_value', 'item_total').show(3)

## 3. Data quality results

Every build appends its checks here. `warn` failures are real quirks of the Olist data, documented rather than hidden.

In [ ]:
checks = spark.read.format('delta').load(lake_path('_quality', 'silver_checks'))
latest = checks.agg(F.max('run_id')).first()[0]
(checks.filter(F.col('run_id') == latest)
       .select('table', 'check', 'severity', 'failing_rows', 'passed')
       .orderBy('passed', 'table').show(50, truncate=False))

## 4. Ask the enriched table real questions

One row per order item, with customer, product (in English), seller, locations, payment and review joined in.

In [ ]:
e = silver('order_items_enriched')

print('Top 10 categories by revenue')
(e.groupBy('product_category_name_english')
  .agg(F.sum('price').alias('revenue'), F.countDistinct('order_id').alias('orders'))
  .orderBy(F.desc('revenue')).show(10, truncate=False))

In [ ]:
print('Does late delivery hurt review scores?')
(e.filter(F.col('is_delivered_late').isNotNull())
  .dropDuplicates(['order_id'])
  .groupBy('is_delivered_late')
  .agg(F.round(F.avg('order_review_score'), 2).alias('avg_review'), F.count('*').alias('orders'))
  .show())

## All green? Phase 4 is done.

Next: Phase 5, Gold marts built from these tables, and incremental (MERGE) loads instead of full rebuilds.